# Выполнение ЛР №4: Иерархическая и вероятностная кластеризации

## Подключение библиотек

In [ ]:
import warnings
import pandas               as pd
import numpy                as np

import matplotlib.pyplot    as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score

from scipy.cluster.hierarchy import dendrogram, linkage



## Настройка библиотек

In [ ]:
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_rows', None)

warnings.filterwarnings('ignore')

## Задание 1

### Формулировка

* Изучить датасеты
* Определить, нужно ли проводить 
нормализацию имеющихся данных?
* Выполнить её при необходимости

### Решение

In [ ]:
#### Изучение датасетов

# Загрузка первого датасета (flame.txt)
flame_data = pd.read_csv('Вариант 4/flame.txt', sep='\t', header=None, names=['x', 'y', 'cluster'])
print("Первый датасет (flame.txt):")
print(f"Размер: {flame_data.shape}")
print(f"Столбцы: {flame_data.columns.tolist()}")
print("\nПервые 10 строк:")
display(flame_data.head(10))
print("\nСтатистическое описание:")
display(flame_data.describe())

# Загрузка второго датасета (Seed_Data.csv)
seed_data = pd.read_csv('Вариант 4/Seed/Seed_Data.csv')
print("\n" + "="*50)
print("Второй датасет (Seed_Data.csv):")
print(f"Размер: {seed_data.shape}")
print(f"Столбцы: {seed_data.columns.tolist()}")
print("\nПервые 10 строк:")
display(seed_data.head(10))
print("\nСтатистическое описание:")
display(seed_data.describe())

In [ ]:
#### Анализ необходимости нормализации

# Анализ масштабов данных в первом датасете
print("\n" + "="*50)
print("АНАЛИЗ НЕОБХОДИМОСТИ НОРМАЛИЗАЦИИ")
print("\nПервый датасет (flame.txt):")
print(f"Диапазон X: {flame_data['x'].min():.2f} - {flame_data['x'].max():.2f}")
print(f"Диапазон Y: {flame_data['y'].min():.2f} - {flame_data['y'].max():.2f}")
print(f"Стандартное отклонение X: {flame_data['x'].std():.2f}")
print(f"Стандартное отклонение Y: {flame_data['y'].std():.2f}")

# Анализ масштабов данных во втором датасете
print("\nВторой датасет (Seed_Data.csv):")
features = seed_data.drop('target', axis=1)
for col in features.columns:
    print(f"{col}: диапазон {features[col].min():.3f} - {features[col].max():.3f}, std = {features[col].std():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# График первого датасета
axes[0].scatter(flame_data['x'], flame_data['y'], c=flame_data['cluster'])
axes[0].set_title('Первый датасет (flame.txt) - исходные данные')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].grid(True, alpha=0.3)

# График второго датасета (первые два признака)
scatter = axes[1].scatter(seed_data['A'], seed_data['C'], c=seed_data['target'], alpha=0.7)
axes[1].set_title('Второй датасет (Seed) - Area vs Perimeter')
axes[1].set_xlabel('Area (A)')
axes[1].set_ylabel('Compactness (C)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#### Выполнение нормализации

# Для первого датасета
print("\n" + "="*50)
print("НОРМАЛИЗАЦИЯ ДАННЫХ")

# Первый датасет - координаты имеют схожие масштабы, но для кластеризации лучше нормализовать
flame_features = flame_data[['x', 'y']].copy()
scaler_flame = StandardScaler()
flame_normalized = scaler_flame.fit_transform(flame_features)
flame_data_norm = pd.DataFrame(flame_normalized, columns=['x_norm', 'y_norm'])
flame_data_norm['cluster'] = flame_data['cluster']

print("Первый датасет после нормализации:")
display(flame_data_norm.describe())

# Второй датасет - признаки имеют разные масштабы, нормализация необходима
seed_features = seed_data.drop('target', axis=1)
scaler_seed = StandardScaler()
seed_normalized = scaler_seed.fit_transform(seed_features)
seed_data_norm = pd.DataFrame(seed_normalized, columns=[f'{col}_norm' for col in seed_features.columns])
seed_data_norm['target'] = seed_data['target']

print("\nВторой датасет после нормализации:")
display(seed_data_norm.describe())

In [ ]:
# Визуализация нормализованных данных
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# График первого датасета после нормализации
axes[0].scatter(flame_data_norm['x_norm'], flame_data_norm['y_norm'], 
               c=flame_data_norm['cluster'])
axes[0].set_title('Первый датасет - нормализованные данные')
axes[0].set_xlabel('X (нормализованный)')
axes[0].set_ylabel('Y (нормализованный)')
axes[0].grid(True, alpha=0.3)

# График второго датасета после нормализации
axes[1].scatter(seed_data_norm['A_norm'], seed_data_norm['C_norm'], 
               c=seed_data_norm['target'], alpha=0.7)
axes[1].set_title('Второй датасет - нормализованные данные')
axes[1].set_xlabel('Area (нормализованная)')
axes[1].set_ylabel('Compactness (нормализованный)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Вывод

1. Первый датасет (flame.txt):
   - Содержит 240 точек с координатами (x, y) и метками кластеров
   - Координаты имеют схожие масштабы, но нормализация улучшит качество кластеризации
   - Данные представляют собой двумерные точки, образующие характерную форму

2. Второй датасет (Seed_Data.csv):
   - Содержит 210 образцов зерен пшеницы с 7 геометрическими признаками
   - Признаки имеют существенно разные масштабы (от 0.87 до 16.63)
   - Нормализация РЕКОМЕНДУЕТСЯ для корректной работы алгоритмов кластеризации
   - Данные содержат 3 класса (сорта пшеницы)

3. Нормализация выполнена методом StandardScaler для обоих датасетов
   - Все признаки приведены к стандартному нормальному распределению (μ=0, σ=1)
   - Это обеспечит равный вклад всех признаков в процесс кластеризации

## Задание 2 (Датасет #1)

### Формулировка

Для первого датасета:
* Определить число кластеров, построив дендрограмму.

### Решение

#### Построение дендрограмм с разными методами связывания

In [ ]:
flame_features_norm = flame_data_norm[["x_norm", "y_norm"]]
methods = ['ward', 'complete', 'average', 'single']
metrics = ['euclidean', 'manhattan', 'chebyshev', 'hamming']
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

linkage_matrices = {}

for i, method in enumerate(methods):
    # Вычисление матрицы связей
    linkage_matrix = linkage(flame_features_norm, method=method, metric=metrics[0])
    
    linkage_matrices[method] = linkage_matrix
    
    # Построение дендрограммы
    dendrogram(linkage_matrix, ax=axes[i], truncate_mode='lastp', p=20, leaf_font_size=8)
    axes[i].set_title(f'Дендрограмма (метод: {method})')
    axes[i].set_xlabel('Индекс кластера или размер кластера')
    axes[i].set_ylabel('Расстояние')

plt.tight_layout()
plt.show()

#### Детальная дендрограмма для метода Ward (наиболее подходящий для данной задачи)

In [ ]:
plt.figure(figsize=(20, 8))

# Построение полной дендрограммы
ward_linkage = linkage_matrices['ward']
dendrogram(ward_linkage, leaf_rotation=90, leaf_font_size=5)
plt.title('Полная дендрограмма (метод Ward)', fontsize=14)
plt.xlabel('Индекс образца')
plt.ylabel('Расстояние')

plt.show()

#### Добавляет линии порогов для 2х и 3х кластеров

In [ ]:
ward_linkage = linkage_matrices['ward']
plt.figure(figsize=(20, 8))
dendrogram(ward_linkage, leaf_rotation=90, leaf_font_size=5)
plt.title('Полная дендрограмма (метод Ward)', fontsize=14)
plt.xlabel('Индекс образца')
plt.ylabel('Расстояние')
plt.axhline(y=16, color='red', linestyle='--', alpha=0.7, label='Порог для 2 кластеров')
plt.axhline(y=13, color='orange', linestyle='--', alpha=0.7, label='Порог для 3 кластеров')

plt.legend()
plt.show()

#### Выводы

1. АНАЛИЗ ДЕНДРОГРАММЫ:
    - Наибольшие скачки на последних этапах слияния можно наблюдать в методах `Complete` и `Ward`
    - Метод `Ward` показал наилучшие результаты для данного датасета
    - Дендрограмма показывает два или три основных кластера

## Задание 3 (Датасет #1)

### Формулировка

Для первого датасета:
* Построить  график  зависимости  расстояний  между кластерами от шага слияния. 
* Определить по графику оптимальное число кластеров.

### Решение

In [ ]:
# Извлекаем расстояния слияния из матрицы связей
ward_linkage = linkage_matrices['ward']
distances = ward_linkage[:, 2]

#### График зависимости расстояний от шага слияния

In [ ]:
# Создаем массив шагов слияния (от 1 до n-1, где n - количество точек)
steps = np.arange(1, len(distances) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# График 1: Полный график всех шагов слияния
axes[0].plot(steps, distances, 'b-', linewidth=2, marker='o', markersize=3)
axes[0].set_title('Расстояния слияния на всех шагах', fontsize=12)
axes[0].set_xlabel('Шаг слияния')
axes[0].set_ylabel('Расстояние между кластерами')
axes[0].grid(True, alpha=0.3)

# График 2: Последние 30 шагов (наиболее важные для определения числа кластеров)
last_steps = 30
axes[1].plot(steps[-last_steps:], distances[-last_steps:], 'r-', linewidth=2, marker='o', markersize=5)
axes[1].set_title(f'Последние {last_steps} шагов слияния', fontsize=12)
axes[1].set_xlabel('Шаг слияния')
axes[1].set_ylabel('Расстояние между кластерами')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Графики приращений расстояний

In [ ]:
# Вычисляем разницу между последовательными расстояниями
distance_diffs = np.diff(distances)

# График приращений
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# График 1: Все приращения
axes[0].plot(range(2, len(distances) + 1), distance_diffs, 'g-', linewidth=2, marker='o', markersize=3)
axes[0].set_title('Приращения расстояний между шагами', fontsize=12)
axes[0].set_xlabel('Шаг слияния')
axes[0].set_ylabel('Приращение расстояния')
axes[0].grid(True, alpha=0.3)

# График 2: Последние приращения (в обратном порядке - от большего к меньшему числу кластеров)
last_steps_diff = 15
x_clusters = np.arange(last_steps_diff, 0, -1)
y_diffs = distance_diffs[-last_steps_diff:]

axes[1].plot(x_clusters, y_diffs, 'r-', linewidth=2, marker='o', markersize=6)
axes[1].set_title('Приращения расстояний (последние шаги)', fontsize=12)
axes[1].set_xlabel('Количество кластеров')
axes[1].set_ylabel('Приращение расстояния')
axes[1].grid(True, alpha=0.3)
axes[1].invert_xaxis()  # Инвертируем ось X для удобства чтения

# Выделяем наибольшие скачки
max_diff_idx = np.argmax(distance_diffs[-last_steps_diff:])
axes[1].axvline(x=x_clusters[max_diff_idx], color='orange', linestyle='--', 
                alpha=0.7, label=f'Максимальный скачок ({x_clusters[max_diff_idx]} кластеров)')
axes[1].legend()

plt.tight_layout()
plt.show()

#### Определение оптимального числа кластеров методом "локтя"

In [ ]:
# Визуализация оптимального числа кластеров
plt.figure(figsize=(14, 6))

# Строим график в обратном порядке (от многих кластеров к малому числу)
n_points = 25
clusters_range = np.arange(n_points, 0, -1)
distances_subset = distances[-n_points:]

plt.plot(clusters_range, distances_subset, 'b-', linewidth=2.5, marker='o', markersize=7)
plt.title('Определение оптимального числа кластеров (метод "локтя")', fontsize=14)
plt.xlabel('Количество кластеров', fontsize=12)
plt.ylabel('Расстояние слияния', fontsize=12)
plt.grid(True, alpha=0.3)

# Отмечаем потенциальные оптимальные точки
optimal_candidates = [2, 3, 4]
colors = ['red', 'orange', 'green']
for k, color in zip(optimal_candidates, colors):
    idx = n_points - k
    plt.scatter(k, distances_subset[idx], s=200, c=color, alpha=0.8, 
                edgecolors='black', linewidth=2, zorder=5,
                label=f'{k} кластера')

plt.gca().invert_xaxis()
plt.legend()
plt.tight_layout()
plt.show()

### Вывод

**Анализ графиков:**

1. **График расстояний слияния** показывает, что наибольшие скачки происходят на последних этапах слияния, когда количество кластеров уменьшается с 4 до 3 и с 2 до 1.

2. **График приращений расстояний** четко демонстрирует "локти" - точки, где приращение резко возрастает:
   - Самый большой скачок наблюдается при переходе от 2 к 1 кластеру
   - Второй значительный скачок - при переходе от 4 к 3 кластерам

3. **Оптимальное число кластеров:**
   - **4 кластера** - наиболее очевидный выбор, так как это на следующем этапе слияния наблюдается первый существенный скачок.
   - **2 кластера** - также хороший вариант, учитывая что он по величине приращения является наибольшим. Стоит учитывать, что слияние от 3 кластеров к 2 не дало значительного приращения расстояния, что свидетельствует о формировании неестественной группы. 
   
4. **Рекомендация:** Для датасета flame, на основе графика зависимости расстояния между кластерами от шага слияния, оптимальным является **4 кластера**. Окончательный выбор будет сделан в следующем задании.

## Задание 4 (Датасет #1) <a id="mb_z4">

### Формулировка

Для первого датасета:
* Выбрать  оптимальный  алгоритм  и  выполнить 
кластеризацию.  
* Пробовать  разные  варианты,  чтобы определить наиболее подходящий вариант. 

### Решение

#### Определение алгоритмов и параметров для тестирования

In [ ]:

# Используем нормализованные данные из задания 1
X_flame = flame_data_norm[['x_norm', 'y_norm']].values
true_labels = flame_data['cluster'].values

# Параметры для тестирования
n_clusters_list = [2 ,4]
algorithms = {
    'K-Means': None,
    'Agglomerative_Ward': 'ward',
    'Agglomerative_Single': 'single', 
    'Agglomerative_Complete': 'complete',
    'Agglomerative_Average': 'average'
}
metrics = ['euclidean', 'manhattan', 'chebyshev', 'hamming']


In [ ]:
# Функция для вычисления внутренних метрик качества
def calculate_internal_metrics(X, labels):
    """Вычисляет компактность и отделимость кластеров"""
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels)
    
    if n_clusters < 2:
        return 0, 0
    
    # Компактность (внутрикластерная дисперсия)
    within_cluster_ss = 0
    cluster_centers = []
    
    for label in unique_labels:
        cluster_points = X[labels == label]
        if len(cluster_points) > 0:
            center = np.mean(cluster_points, axis=0)
            cluster_centers.append(center)
            within_cluster_ss += np.sum((cluster_points - center) ** 2)
    
    # Отделимость (межкластерная дисперсия)
    overall_center = np.mean(X, axis=0)
    between_cluster_ss = 0
    
    for i, label in enumerate(unique_labels):
        cluster_size = np.sum(labels == label)
        if cluster_size > 0:
            between_cluster_ss += cluster_size * np.sum((cluster_centers[i] - overall_center) ** 2)
    
    return within_cluster_ss, between_cluster_ss


def get_alg_result(alg_name, n_clusters, X_flame, true_labels, cluster_labels, other):
    # Вычисление метрик качества
    rand_score = adjusted_rand_score(true_labels, cluster_labels)
    #silhouette = silhouette_score(X_flame, cluster_labels)
    within_ss, between_ss = calculate_internal_metrics(X_flame, cluster_labels)
    sum_ss = between_ss - within_ss
    
     # Сохранение результатов
    result = {
        'algorithm': alg_name,
        'n_clusters': n_clusters,
        'rand_score': rand_score,
        #'silhouette_score': silhouette,
        'within_cluster_ss': within_ss,
        'between_cluster_ss': between_ss,
        'summ_cluster_ss': sum_ss,
        'labels': cluster_labels,
        'other': other
    }
    return result

#### Выполнение кластеризации всеми алгоритмами

In [ ]:

results = []

for n_clusters in n_clusters_list:
    for alg_name, linkage_method in algorithms.items():  
        # Выполнение кластеризации
        if alg_name == 'K-Means':
            clusterer = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            cluster_labels = clusterer.fit_predict(X_flame)
            results.append(get_alg_result(alg_name, n_clusters, X_flame, true_labels, cluster_labels,'lloyd'))
        else:
            # Агломеративная кластеризация
            if linkage_method == 'ward':
                _metrics = ['euclidean'] 
            else: 
                _metrics = metrics         
            for metric in _metrics:
                clusterer = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage_method, metric=metric)
                cluster_labels = clusterer.fit_predict(X_flame)
                results.append(get_alg_result(alg_name, n_clusters, X_flame, true_labels, cluster_labels,metric))
            

#### Анализ результатов и выбор лучших алгоритмов

In [ ]:
# Создание DataFrame с результатами
results_df = pd.DataFrame([{
    'Алгоритм': r['algorithm'],
    'Параметр': r['other'],
    'Кластеры': r['n_clusters'],
    'Индекс Рэнда': r['rand_score'],
    #'Силуэт': r['silhouette_score'],
    'Внутр. SS': r['within_cluster_ss'],
    'Межкл. SS': r['between_cluster_ss'],
    'Cумм. SS': r['summ_cluster_ss']
} for r in results])

print("\n" + "=" * 80)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:")
print("=" * 80)
display(results_df.round(3))


# Выбор топ-3 алгоритмов по индексу Рэнда
print("\n" + "=" * 80)
print("Топ-3 алгоритмов по индексу Рэнда:")
print("=" * 80)

top_algorithms_rand = results_df.nlargest(3, 'Индекс Рэнда')
for idx, row in top_algorithms_rand.iterrows():
    print(f"{row['Алгоритм']} ({row['Кластеры']} кластеров): {row['Индекс Рэнда']:.3f}")


# # Выбор топ-3 алгоритмов по силуэту
# print("\n" + "=" * 80)
# print("Топ-3 алгоритмов по силуэту:")
# print("=" * 80)

# top_algorithms_silhouette = results_df.nlargest(3, 'Силуэт')
# for idx, row in top_algorithms_silhouette.iterrows():
#     print(f"{row['Алгоритм']} ({row['Кластеры']} кластеров): {row['Силуэт']:.3f}")


# Выбор топ-3 алгоритмов по дисперсии
print("\n" + "=" * 80)
print("Топ-3 алгоритмов по дисперсии:")
print("=" * 80)

top_algorithms_ss = results_df.nlargest(3, 'Cумм. SS')
for idx, row in top_algorithms_ss.iterrows():
    print(f"{row['Алгоритм']} ({row['Кластеры']} кластеров): {row['Cумм. SS']:.3f}")

# Выбор лучших алгоритмов по каждой метрике
print("\n" + "=" * 80)
print("Лучшие алгоритмы по каждой метрике:")
print("=" * 80)

best_rand = results_df.loc[results_df['Индекс Рэнда'].idxmax()]
#best_silhouette = results_df.loc[results_df['Силуэт'].idxmax()]
best_ss = results_df.loc[results_df['Cумм. SS'].idxmax()]

print(f"По индексу Рэнда: {best_rand['Алгоритм']} ({best_rand['Кластеры']} кластеров) = {best_rand['Индекс Рэнда']:.3f}")
#print(f"По силуэту: {best_silhouette['Алгоритм']} ({best_silhouette['Кластеры']} кластеров) = {best_silhouette['Силуэт']:.3f}")
print(f"По кластерной дисперсии: {best_ss['Алгоритм']} ({best_ss['Кластеры']} кластеров) = {best_ss['Cумм. SS']:.3f}")

#### Визуализация результатов лучших алгоритмов по внешним метрикам

In [ ]:
# Подготовка данных для визуализации
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

# Исходные данные с истинными метками
axes[0].scatter(flame_data['x'], flame_data['y'], c=true_labels, cmap='viridis', alpha=0.7)
axes[0].set_title('Истинные кластеры (исходные данные)', fontsize=12)
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].grid(True, alpha=0.3)

# Визуализация топ-3 алгоритмов
for i, (idx, row) in enumerate(top_algorithms_rand.iterrows()):
    
    # Находим соответствующие метки кластеров
    result = next(r for r in results if r['algorithm'] == row['Алгоритм'] and r['n_clusters'] == row['Кластеры'] and r['other'] == row['Параметр'])
    labels = result['labels']
    
    axes[i+1].scatter(flame_data['x'], flame_data['y'], c=labels, cmap='viridis', alpha=0.7)
    axes[i+1].set_title(f'{row["Алгоритм"]} ({row["Кластеры"]} кластеров)\n Параметр: {row["Параметр"]}\nРэнд: {row["Индекс Рэнда"]:.3f}', fontsize=12)
    axes[i+1].set_xlabel('X')
    axes[i+1].set_ylabel('Y')
    axes[i+1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Визуализация результатов лучших алгоритмов по внутренним метриками

In [ ]:
# Подготовка данных для визуализации
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

# Исходные данные с истинными метками
axes[0].scatter(flame_data['x'], flame_data['y'], c=true_labels, cmap='viridis', alpha=0.7)
axes[0].set_title('Истинные кластеры (исходные данные)', fontsize=12)
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].grid(True, alpha=0.3)

# Визуализация топ-3 алгоритмов 
for i, (idx, row) in enumerate(top_algorithms_ss.iterrows()):
    
    # Находим соответствующие метки кластеров
    result = next(r for r in results if r['algorithm'] == row['Алгоритм'] and r['n_clusters'] == row['Кластеры'] and r['other'] == row['Параметр'])
    labels = result['labels']
    
    axes[i+1].scatter(flame_data['x'], flame_data['y'], c=labels, cmap='viridis', alpha=0.7)
    axes[i+1].set_title(f'{row["Алгоритм"]} ({row["Кластеры"]} кластеров)\n Параметр: {row["Параметр"]}\nCумм. SS: {row["Cумм. SS"]:.3f}', fontsize=12)
    axes[i+1].set_xlabel('X')
    axes[i+1].set_ylabel('Y')
    axes[i+1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Выводы по заданию 4

**Анализ результатов кластеризации первого датасета:**

* **Оптимальное число кластеров:** 4
   - Существенный рост межкластерной дисперсии (особенно у K-Means и Complete).
   - Индекс Рэнда остаётся высоким: Agglomerative Average (k=4): 0.635
   - Кластеры становятся более компактными и интерпретируемыми.
   - Максимальный Rand чуть ниже абсолютного лидера при k=2, но разница минимальна и компенсируется лучшей структурой.

* **Лучшие алгоритмы:**
   
   1. K-Means (k = 4)
      -  Лучшая кластерная дисперсия: 255.072
      -  Хороший индекс Рэнда: 0.426
      -  Чёткое разделение и компактные кластеры.

   1. Agglomerative Complete (k = 4, euclidean / chebyshev)
      -  Высокая межкластерная дисперсия: до 219.270
      -  Более строгие границы кластеров, чем у Average.
      -  Устойчив к «растягиванию» кластеров.

   1. Agglomerative Average (k = 4)
      - Почти максимальный Rand: 0.635
      - Хороший компромисс между качеством и структурой.

* **Исключение алгоритмов** (крайне низкий Rand, почти нулевая межкластерная SS, эффект «цепочки».):
   - Agglomerative
   - Метрика расстояния Hamming 

## Задание 5 (Датасет #2)

### Формулировка

Для прикладного датасета (второй)
* Определить число кластеров 
    * методом расчёта сумм расстояний от точек данных до центра  ближайшего  к  ней  кластера  (инерций).  
* Для кластеризации используйте как исходный датасет без  предобработки,  так  и  с  предобработкой, выполненной в лабораторной работе 3. 

### Решение

## Задание 6 (Датасет #2) <a id="mb_z6">

### Формулировка

Для прикладного датасета (второй)
*  Выполнить  кластеризацию  методом  k-средних.

### Решение

## Задание 7 (Датасет #1,2)

### Формулировка

* Провести анализ полученных результатов из заданий [4](#mb_z4) и 
[6](#mb_z6)
* Cформулировать отличительные особенности разных 
кластеров для прикладного датасета.

### Решение